# Jacobian-lens layer-by-token preview with PrivacyGuard
For questions, reach out to Matthew Grange

This tutorial loads a `privacyguard.jlens_preview/v1` receipt, summarizes it with the optional, experimental
`JacobianLensPreviewNode`, and renders a self-contained interactive layer-by-token viewer.

The example is the prompt `The number of legs on the animal that spins webs is`. The model's own output is the
reference: a Jacobian lens (J-lens) and an ordinary logit lens read out ranked vocabulary tokens at every
(layer, token position) cell, and the viewer tracks the ranks of `spider`, `ant`, `8` and `6`.

What this analysis does **not** claim: it says nothing about model intent, a global workspace, consciousness or
privacy efficacy. J-lens layers are candidate sites, not validated workspace bands. A readout is not a causal
result; the spider-to-ant intervention and its controls are reported separately, including negative results.

## Environment set-up

Install PrivacyGuard from a clone of the repository. The preview needs only the base dependencies
(`torch`, `pandas`, `matplotlib`); it loads no model and makes no network call.

In [ ]:
# From the repository root:
# !pip install -e .

In [ ]:
import gzip
import json
import os

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from privacy_guard.analysis.jacobian_lens.preview_analysis_input import JacobianLensPreviewInput
from privacy_guard.analysis.jacobian_lens.preview_analysis_node import JacobianLensPreviewNode
from privacy_guard.analysis.jacobian_lens.preview_schema import JLensPreviewReceipt
from privacy_guard.analysis.jacobian_lens.preview_viewer import HTML_FILENAME, write_preview_bundle

## Load and validate the receipts

A receipt file (optionally gzip-compressed) holds a JSON list with one receipt per prompt. Loading validates the schema,
the pinned identity fields, rank consistency, the receipt hash, and rejects any activation, residual, logit array
or weight payload. A receipt whose `run_kind` is not `real_model` is a fixture and is labeled as such everywhere.

In [ ]:
RECEIPTS_PATH = os.environ.get("JLENS_RECEIPTS_PATH", "data/jacobian_lens/jlens_preview_receipts.json.gz")

opener = gzip.open if RECEIPTS_PATH.endswith(".gz") else open
with opener(RECEIPTS_PATH, "rt", encoding="utf-8") as f:
    receipts = [JLensPreviewReceipt.from_dict(r) for r in json.load(f)]

pd.DataFrame(
    [
        {
            "prompt": r.prompt.label or r.prompt.text,
            "run_kind": r.run_kind,
            "model_role": r.model_role,
            "repo_id": r.model_identity["repo_id"],
            "revision_sha": r.model_identity["revision_sha"],
            "dtype": r.model_identity["dtype"],
            "lens_id": None if r.lens is None else r.lens["lens_id"],
            "score_definition": r.score_definition,
            "receipt_sha256": r.receipt_sha256,
        }
        for r in receipts
    ]
)

## Summarize with the analysis node

`head_consistency` checks that the final-layer logit lens at the last prompt position reproduces the model's
actual next-token scores. `answer_margins` compares the best `6` score with the best `8` score at the next token,
for the actual output and for each intervention or control run.

In [ ]:
HEAD_SCORE_TOLERANCE = 0.125  # one BF16 step at |score| in [16, 32)
summaries = [
    JacobianLensPreviewNode(JacobianLensPreviewInput(r), head_score_tolerance=HEAD_SCORE_TOLERANCE).compute_outputs()
    for r in receipts
]
for receipt, summary in zip(receipts, summaries):
    print(receipt.prompt.label or receipt.prompt.text)
    print("  labels:", *summary["labels"], sep="\n    ")
    print("  actual next token:", summary["actual_next_token"], "greedy:", repr(summary["actual_greedy_text"]))
    print("  head consistency:", summary["head_consistency"])
    display(pd.DataFrame(summary["answer_margins"]))

## Rank curves at the last prompt position

Each line is the best full-vocabulary rank over a concept's tracked token variants (for example `" spider"` and
`"Spider"`). Solid lines are the J-lens, dashed lines the logit lens. Missing points are unavailable cells.

In [ ]:
for receipt, summary in zip(receipts, summaries):
    last = len(receipt.prompt.token_ids) - 1
    fig, ax = plt.subplots(figsize=(8, 3.5))
    for curve in summary["rank_curves"]:
        if curve["position"] != last:
            continue
        points = [(layer, rank) for layer, rank in zip(curve["layers"], curve["best_rank"]) if rank is not None]
        if points:
            ax.plot(*zip(*points), linestyle="-" if curve["method"] == "jlens" else "--", label=f"{curve['concept']} ({curve['method']})")
    ax.set_yscale("log")
    ax.invert_yaxis()
    ax.set_xlabel("layer (block output, 0-based)")
    ax.set_ylabel("full-vocabulary rank")
    ax.set_title(receipt.prompt.label or receipt.prompt.text)
    ax.legend(fontsize=7, ncol=2)
    plt.show()

## Tracked-token ranks as a table

`JacobianLensPreviewInput.tracked_ranks` is a long table with one row per (method, layer, position, tracked token).

In [ ]:
table = JacobianLensPreviewInput(receipts[0]).tracked_ranks
last = len(receipts[0].prompt.token_ids) - 1
table[(table["source"] == "receipt") & (table["position"] == last)].pivot_table(
    index="layer", columns=["method", "concept"], values="rank", aggfunc="min"
)

## Interactive layer-by-token viewer

The next cell writes the viewer next to this notebook (it is not stored in the repository: the page embeds every
receipt, about 15 MB here). The viewer is one HTML file with inlined data, CSS and JavaScript and a Content-Security-Policy that blocks all
network requests. Click a cell for the ranked vocabulary (raw token IDs, decoded text, scores and full-vocabulary
ranks), J-lens next to logit lens next to the actual output, tracked-token ranks and rank curves. The optional
word-only display filter is labeled and never changes the reported ranks.

In [ ]:
output_dir = "jlens_preview_output"
digests = write_preview_bundle(receipts, output_dir, head_score_tolerance=HEAD_SCORE_TOLERANCE)
for name, digest in sorted(digests.items()):
    print(digest, name)
print(f"Open {os.path.join(output_dir, HTML_FILENAME)} in a browser; it needs no network access.")


## Regenerating the receipts

The bundled receipts come from the public `meta-models/Muse-Glimmer-30B` main BF16 checkpoint at the revision pinned in
each receipt's `model_identity` (its shard and tokenizer SHA-256 values are verified before loading). Regenerating them
needs one GPU with about 80 GB of memory and the checkpoint downloaded locally; the command each receipt was produced
with is recorded, with local paths replaced by placeholders, in its `code.run_command`:

In [ ]:
for receipt in receipts:
    print(receipt.prompt.label, "|", receipt.code.get("run_command"))


Render a bundle from any receipts file without a GPU (the actual output comes from the native BF16 forward pass while cell
scores use the head in float32, so the final-layer consistency check allows one BF16 step, 0.125):

```
python -m privacy_guard.analysis.jacobian_lens.preview_render_main \
    --receipts data/jacobian_lens/jlens_preview_receipts.json.gz --out-dir jlens_preview_output --head-score-tolerance 0.125
```

The bundled file is trimmed for size (shorter top lists, rounded scores, patched-pass readouts kept for the primary arm
only); `display.trimmed` in each receipt records the SHA-256 of the full receipt it came from.

## Attribution

- Jacobian lens method and estimator: Anthropic, `jacobian-lens` reference implementation at commit
  `581d398613e5602a5af361e1c34d3a92ea82ba8e` (Apache License 2.0).
- Model: `meta-models/Muse-Glimmer-30B` main BF16 checkpoint (model card license Apache-2.0); the exact revision
  and file hashes are in each receipt's `model_identity`.